In [ ]:
import os
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

dataset = "drawbench"
cfg_guidance = 1.0
# method = "SD3.5M-DiffusionNFT-MultiReward"
method = "sd3.5m-diffusionnft-multireward-next-code-geneval-lr_0.0001-resize_256_crop_224-decay_type_7_0.01"
# method = "sd3.5m-diffusionnft-multireward-next-dinov2-geneval-lr_0.0001-resize_256_crop_224-decay_type_7_0.01-max_min_threshold_0.4"
ckpt=5
base_npy_dir = "/data_center/data2/dataset/chenwy/21164-data/Artifact_Segmentor_Tmp"
base_image_dir = f"/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_{cfg_guidance}/{dataset}"
base_target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/Artifact_Segmentor_Tmp/visualization/ImageDoctor"

npy_dir = os.path.join(base_npy_dir, dataset, method)
image_dir = os.path.join(base_image_dir, method, f"ckpt-{ckpt}", "images")
target_image_dir = os.path.join(base_target_image_dir, dataset, method)
os.makedirs(target_image_dir, exist_ok=True)

print(f"image_dir: {image_dir}")
def apply_heatmap_to_image(image_path, npy_path, output_path, alpha=0.5):
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    
    heatmap_data = np.load(npy_path)
    
    heatmap_resized = heatmap_data
    
    heatmap_colored = plt.cm.jet(heatmap_resized)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)
    
    overlay = cv2.addWeighted(image_np, 1 - alpha, heatmap_colored, alpha, 0)
    
    result_image = Image.fromarray(overlay)
    result_image.save(output_path)
    return output_path

def process_all_images():
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', ".JPEG", ".PNG"}
    image_files = [f for f in os.listdir(image_dir) 
                   if os.path.splitext(f)[1].lower() in image_extensions]
    
    print(f"Total image_files num: {len(image_files)}")
    
    processed_count = 0
    
    for image_file in tqdm(image_files[0:1000]):
        image_path = os.path.join(image_dir, image_file)
        
        base_name = os.path.splitext(image_file)[0]
        npy_file = base_name + '.npy'
        npy_path = os.path.join(npy_dir, npy_file)
        
        output_path = os.path.join(target_image_dir, f"{base_name}.png")
    
        apply_heatmap_to_image(image_path, npy_path, output_path, alpha=0.4)
        processed_count += 1

process_all_images()